# Gold Analytics — Vistas para Dashboard

Este notebook crea la capa semántica del Observatorio Energético en:

`observatorio_dev.gold_analytics`

Debe ejecutarse después de `gold_daily` y antes de `quality_check`.

Las vistas se construyen únicamente sobre las tablas Gold validadas.


## 1. Configuración y validación de tablas

In [0]:
from pyspark.sql import functions as F

CATALOG = "observatorio_dev"
GOLD_SCHEMA = f"{CATALOG}.gold"
ANALYTICS_SCHEMA = f"{CATALOG}.gold_analytics"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {ANALYTICS_SCHEMA}"
)

required_tables = [
    f"{GOLD_SCHEMA}.dim_fecha",
    f"{GOLD_SCHEMA}.dim_periodo",
    f"{GOLD_SCHEMA}.dim_agente",
    f"{GOLD_SCHEMA}.dim_planta",
    f"{GOLD_SCHEMA}.dim_embalse",
    f"{GOLD_SCHEMA}.bridge_planta_embalse",
    f"{GOLD_SCHEMA}.fact_generacion_real",
    f"{GOLD_SCHEMA}.fact_disponibilidad_planta",
    f"{GOLD_SCHEMA}.fact_demanda_real",
    f"{GOLD_SCHEMA}.fact_precio_bolsa",
    f"{GOLD_SCHEMA}.fact_energia_embalsada_planta",
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]

print("Esquema analítico:", ANALYTICS_SCHEMA)
print("Tablas faltantes:", missing_tables)

if missing_tables:
    raise ValueError(
        f"Faltan tablas Gold requeridas: {missing_tables}"
    )

print("Todas las tablas Gold requeridas están disponibles.")

## 2. Vista horaria del sistema

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_sistema_horario AS

WITH horas AS (
    SELECT fecha_hora
    FROM observatorio_dev.gold.fact_generacion_real

    UNION

    SELECT fecha_hora
    FROM observatorio_dev.gold.fact_disponibilidad_planta

    UNION

    SELECT fecha_hora
    FROM observatorio_dev.gold.fact_demanda_real

    UNION

    SELECT fecha_hora
    FROM observatorio_dev.gold.fact_precio_bolsa
),

generacion AS (
    SELECT
        fecha_hora,
        MAX(fecha_key) AS fecha_key,
        MAX(periodo_key) AS periodo_key,
        SUM(generacion_real_kwh) AS generacion_real_kwh
    FROM observatorio_dev.gold.fact_generacion_real
    GROUP BY fecha_hora
),

disponibilidad AS (
    SELECT
        fecha_hora,
        MAX(fecha_key) AS fecha_key,
        MAX(periodo_key) AS periodo_key,
        SUM(disponibilidad_real_kwh) AS disponibilidad_real_kwh
    FROM observatorio_dev.gold.fact_disponibilidad_planta
    GROUP BY fecha_hora
),

demanda AS (
    SELECT
        fecha_hora,
        MAX(fecha_key) AS fecha_key,
        MAX(periodo_key) AS periodo_key,
        SUM(demanda_real_kwh) AS demanda_total_kwh,

        SUM(
            CASE
                WHEN UPPER(TRIM(tipo_mercado)) = 'REGULADO'
                THEN demanda_real_kwh
                ELSE 0
            END
        ) AS demanda_regulada_kwh,

        SUM(
            CASE
                WHEN UPPER(TRIM(tipo_mercado)) IN (
                    'NO REGULADO',
                    'NO_REGULADO'
                )
                THEN demanda_real_kwh
                ELSE 0
            END
        ) AS demanda_no_regulada_kwh

    FROM observatorio_dev.gold.fact_demanda_real
    GROUP BY fecha_hora
),

precio AS (
    SELECT
        fecha_hora,
        MAX(fecha_key) AS fecha_key,
        MAX(periodo_key) AS periodo_key,
        MAX(precio_bolsa_nacional_cop_kwh)
            AS precio_bolsa_nacional_cop_kwh,
        MAX(precio_bolsa_internacional_cop_kwh)
            AS precio_bolsa_internacional_cop_kwh,
        MAX(precio_bolsa_tie_cop_kwh)
            AS precio_bolsa_tie_cop_kwh
    FROM observatorio_dev.gold.fact_precio_bolsa
    GROUP BY fecha_hora
)

SELECT
    COALESCE(
        g.fecha_key,
        d.fecha_key,
        disp.fecha_key,
        p.fecha_key,
        CAST(DATE_FORMAT(h.fecha_hora, 'yyyyMMdd') AS INT)
    ) AS fecha_key,

    COALESCE(
        g.periodo_key,
        d.periodo_key,
        disp.periodo_key,
        p.periodo_key,
        CAST(HOUR(h.fecha_hora) + 1 AS TINYINT)
    ) AS periodo_key,

    h.fecha_hora,
    CAST(h.fecha_hora AS DATE) AS fecha,
    YEAR(h.fecha_hora) AS anio,
    MONTH(h.fecha_hora) AS mes_numero,
    HOUR(h.fecha_hora) + 1 AS periodo,

    g.generacion_real_kwh,
    d.demanda_total_kwh,
    d.demanda_regulada_kwh,
    d.demanda_no_regulada_kwh,
    disp.disponibilidad_real_kwh,

    p.precio_bolsa_nacional_cop_kwh,
    p.precio_bolsa_internacional_cop_kwh,
    p.precio_bolsa_tie_cop_kwh,

    g.generacion_real_kwh
        - d.demanda_total_kwh
        AS balance_generacion_demanda_kwh,

    disp.disponibilidad_real_kwh
        - g.generacion_real_kwh
        AS margen_disponibilidad_kwh,

    CASE
        WHEN disp.disponibilidad_real_kwh > 0
        THEN
            100.0
            * g.generacion_real_kwh
            / disp.disponibilidad_real_kwh
    END AS utilizacion_disponibilidad_pct,

    CASE
        WHEN d.demanda_total_kwh > 0
        THEN
            100.0
            * g.generacion_real_kwh
            / d.demanda_total_kwh
    END AS relacion_generacion_demanda_pct,

    CASE
        WHEN d.demanda_total_kwh IS NOT NULL
         AND p.precio_bolsa_nacional_cop_kwh IS NOT NULL
        THEN
            d.demanda_total_kwh
            * p.precio_bolsa_nacional_cop_kwh
    END AS valor_referencia_bolsa_cop

FROM horas h
LEFT JOIN generacion g
    ON h.fecha_hora = g.fecha_hora
LEFT JOIN demanda d
    ON h.fecha_hora = d.fecha_hora
LEFT JOIN disponibilidad disp
    ON h.fecha_hora = disp.fecha_hora
LEFT JOIN precio p
    ON h.fecha_hora = p.fecha_hora

## 3. Resumen diario del sistema

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_resumen_diario_sistema AS

SELECT
    fecha_key,
    fecha,
    MAX(anio) AS anio,
    MAX(mes_numero) AS mes_numero,

    ROUND(
        SUM(generacion_real_kwh) / 1000000,
        3
    ) AS generacion_gwh,

    ROUND(
        SUM(demanda_total_kwh) / 1000000,
        3
    ) AS demanda_gwh,

    ROUND(
        SUM(demanda_regulada_kwh) / 1000000,
        3
    ) AS demanda_regulada_gwh,

    ROUND(
        SUM(demanda_no_regulada_kwh) / 1000000,
        3
    ) AS demanda_no_regulada_gwh,

    ROUND(
        SUM(disponibilidad_real_kwh) / 1000000,
        3
    ) AS disponibilidad_gwh,

    ROUND(
        SUM(balance_generacion_demanda_kwh) / 1000000,
        3
    ) AS balance_generacion_demanda_gwh,

    ROUND(
        SUM(margen_disponibilidad_kwh) / 1000000,
        3
    ) AS margen_disponibilidad_gwh,

    ROUND(
        100.0
        * SUM(generacion_real_kwh)
        / CASE
            WHEN SUM(disponibilidad_real_kwh) = 0
            THEN NULL
            ELSE SUM(disponibilidad_real_kwh)
          END,
        2
    ) AS utilizacion_disponibilidad_pct,

    ROUND(
        AVG(precio_bolsa_nacional_cop_kwh),
        2
    ) AS precio_nacional_promedio_cop_kwh,

    ROUND(
        MIN(precio_bolsa_nacional_cop_kwh),
        2
    ) AS precio_nacional_minimo_cop_kwh,

    ROUND(
        MAX(precio_bolsa_nacional_cop_kwh),
        2
    ) AS precio_nacional_maximo_cop_kwh,

    ROUND(
        AVG(precio_bolsa_internacional_cop_kwh),
        2
    ) AS precio_internacional_promedio_cop_kwh,

    ROUND(
        AVG(precio_bolsa_tie_cop_kwh),
        2
    ) AS precio_tie_promedio_cop_kwh,

    COUNT(DISTINCT fecha_hora) AS horas_con_datos

FROM observatorio_dev.gold_analytics.vw_sistema_horario

GROUP BY
    fecha_key,
    fecha

## 4. Estado de actualización de fuentes

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_actualizacion_fuentes AS

WITH cobertura AS (
    SELECT
        'Generación real' AS fuente,
        MIN(CAST(fecha_hora AS DATE)) AS fecha_minima,
        MAX(CAST(fecha_hora AS DATE)) AS fecha_maxima,
        COUNT(DISTINCT CAST(fecha_hora AS DATE))
            AS dias_disponibles,
        COUNT(*) AS registros
    FROM observatorio_dev.gold.fact_generacion_real

    UNION ALL

    SELECT
        'Demanda real',
        MIN(CAST(fecha_hora AS DATE)),
        MAX(CAST(fecha_hora AS DATE)),
        COUNT(DISTINCT CAST(fecha_hora AS DATE)),
        COUNT(*)
    FROM observatorio_dev.gold.fact_demanda_real

    UNION ALL

    SELECT
        'Disponibilidad de plantas',
        MIN(CAST(fecha_hora AS DATE)),
        MAX(CAST(fecha_hora AS DATE)),
        COUNT(DISTINCT CAST(fecha_hora AS DATE)),
        COUNT(*)
    FROM observatorio_dev.gold.fact_disponibilidad_planta

    UNION ALL

    SELECT
        'Precio de bolsa',
        MIN(CAST(fecha_hora AS DATE)),
        MAX(CAST(fecha_hora AS DATE)),
        COUNT(DISTINCT CAST(fecha_hora AS DATE)),
        COUNT(*)
    FROM observatorio_dev.gold.fact_precio_bolsa

    UNION ALL

    SELECT
        'Energía embalsada',
        MIN(fecha_medicion),
        MAX(fecha_medicion),
        COUNT(DISTINCT fecha_medicion),
        COUNT(*)
    FROM observatorio_dev.gold.fact_energia_embalsada_planta
)

SELECT
    fuente,
    fecha_minima,
    fecha_maxima,
    dias_disponibles,
    registros,
    DATEDIFF(CURRENT_DATE(), fecha_maxima) AS dias_rezago,
    CURRENT_TIMESTAMP() AS fecha_consulta
FROM cobertura

## 5. Demanda diaria por mercado

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_demanda_diaria_mercado AS

WITH base AS (
    SELECT
        demanda.fecha_key,
        CAST(demanda.fecha_hora AS DATE) AS fecha,
        UPPER(TRIM(demanda.tipo_mercado)) AS tipo_mercado,
        demanda.agente_key,
        demanda.fecha_hora,
        demanda.demanda_real_kwh
    FROM observatorio_dev.gold.fact_demanda_real demanda
)

SELECT
    base.fecha_key,
    base.fecha,
    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,
    calendario.semana_anio,
    calendario.dia_semana_numero,
    calendario.dia_semana_nombre,
    calendario.es_fin_semana,

    base.tipo_mercado,

    ROUND(
        SUM(base.demanda_real_kwh) / 1000000,
        3
    ) AS demanda_total_gwh,

    ROUND(
        AVG(base.demanda_real_kwh) / 1000,
        3
    ) AS demanda_promedio_mw,

    ROUND(
        MAX(base.demanda_real_kwh) / 1000,
        3
    ) AS demanda_pico_mw,

    COUNT(DISTINCT base.agente_key)
        AS agentes_con_demanda,

    COUNT(DISTINCT base.fecha_hora)
        AS horas_con_datos

FROM base

LEFT JOIN observatorio_dev.gold.dim_fecha calendario
    ON base.fecha_key = calendario.fecha_key

GROUP BY
    base.fecha_key,
    base.fecha,
    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,
    calendario.semana_anio,
    calendario.dia_semana_numero,
    calendario.dia_semana_nombre,
    calendario.es_fin_semana,
    base.tipo_mercado

## 6. Demanda diaria por agente

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_demanda_diaria_agente AS

SELECT
    demanda.fecha_key,
    CAST(demanda.fecha_hora AS DATE) AS fecha,

    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,

    demanda.agente_key,
    agente.codigo_agente,
    agente.nombre_agente,
    agente.actividad_agente,
    UPPER(TRIM(demanda.tipo_mercado)) AS tipo_mercado,

    ROUND(
        SUM(demanda.demanda_real_kwh) / 1000000,
        3
    ) AS demanda_total_gwh,

    ROUND(
        AVG(demanda.demanda_real_kwh) / 1000,
        3
    ) AS demanda_promedio_mw,

    ROUND(
        MAX(demanda.demanda_real_kwh) / 1000,
        3
    ) AS demanda_pico_mw,

    COUNT(DISTINCT demanda.fecha_hora)
        AS horas_con_datos

FROM observatorio_dev.gold.fact_demanda_real demanda

LEFT JOIN observatorio_dev.gold.dim_fecha calendario
    ON demanda.fecha_key = calendario.fecha_key

LEFT JOIN observatorio_dev.gold.dim_agente agente
    ON demanda.agente_key = agente.agente_key

GROUP BY
    demanda.fecha_key,
    CAST(demanda.fecha_hora AS DATE),
    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,
    demanda.agente_key,
    agente.codigo_agente,
    agente.nombre_agente,
    agente.actividad_agente,
    UPPER(TRIM(demanda.tipo_mercado))

## 7. Operación diaria por planta

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_operacion_diaria_planta AS

WITH generacion AS (
    SELECT
        fecha_key,
        CAST(fecha_hora AS DATE) AS fecha,
        planta_key,
        SUM(generacion_real_kwh) AS generacion_total_kwh,
        AVG(generacion_real_kwh) AS generacion_promedio_kwh,
        MAX(generacion_real_kwh) AS generacion_pico_kwh,
        COUNT(DISTINCT fecha_hora) AS horas_con_generacion
    FROM observatorio_dev.gold.fact_generacion_real
    GROUP BY
        fecha_key,
        CAST(fecha_hora AS DATE),
        planta_key
),

disponibilidad AS (
    SELECT
        fecha_key,
        CAST(fecha_hora AS DATE) AS fecha,
        planta_key,
        SUM(disponibilidad_real_kwh)
            AS disponibilidad_total_kwh,
        AVG(disponibilidad_real_kwh)
            AS disponibilidad_promedio_kwh,
        MAX(disponibilidad_real_kwh)
            AS disponibilidad_pico_kwh,
        COUNT(DISTINCT fecha_hora)
            AS horas_con_disponibilidad
    FROM observatorio_dev.gold.fact_disponibilidad_planta
    GROUP BY
        fecha_key,
        CAST(fecha_hora AS DATE),
        planta_key
),

llaves AS (
    SELECT fecha_key, fecha, planta_key FROM generacion
    UNION
    SELECT fecha_key, fecha, planta_key FROM disponibilidad
)

SELECT
    llaves.fecha_key,
    llaves.fecha,

    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,

    llaves.planta_key,
    planta.codigo_planta,
    planta.nombre_planta,
    planta.codigo_sic_agente,
    planta.tipo_generacion,
    planta.cap_efectiva_neta,
    planta.es_registro_inferido,
    planta.esta_en_maestro_actual,

    ROUND(
        generacion.generacion_total_kwh / 1000000,
        3
    ) AS generacion_total_gwh,

    ROUND(
        disponibilidad.disponibilidad_total_kwh / 1000000,
        3
    ) AS disponibilidad_total_gwh,

    ROUND(
        generacion.generacion_promedio_kwh / 1000,
        3
    ) AS generacion_promedio_mw,

    ROUND(
        disponibilidad.disponibilidad_promedio_kwh / 1000,
        3
    ) AS disponibilidad_promedio_mw,

    ROUND(
        generacion.generacion_pico_kwh / 1000,
        3
    ) AS generacion_pico_mw,

    ROUND(
        disponibilidad.disponibilidad_pico_kwh / 1000,
        3
    ) AS disponibilidad_pico_mw,

    generacion.horas_con_generacion,
    disponibilidad.horas_con_disponibilidad,

    ROUND(
        100.0
        * generacion.generacion_total_kwh
        / CASE
            WHEN disponibilidad.disponibilidad_total_kwh = 0
            THEN NULL
            ELSE disponibilidad.disponibilidad_total_kwh
          END,
        2
    ) AS utilizacion_disponibilidad_pct,

    CASE
        WHEN generacion.generacion_total_kwh IS NULL
         AND disponibilidad.disponibilidad_total_kwh IS NOT NULL
        THEN 'SOLO DISPONIBILIDAD'

        WHEN generacion.generacion_total_kwh IS NOT NULL
         AND disponibilidad.disponibilidad_total_kwh IS NULL
        THEN 'SOLO GENERACION'

        WHEN generacion.generacion_total_kwh
             > disponibilidad.disponibilidad_total_kwh
        THEN 'GENERACION MAYOR A DISPONIBILIDAD'

        ELSE 'CONSISTENTE'
    END AS estado_consistencia

FROM llaves

LEFT JOIN generacion
    ON llaves.fecha_key = generacion.fecha_key
   AND llaves.planta_key = generacion.planta_key

LEFT JOIN disponibilidad
    ON llaves.fecha_key = disponibilidad.fecha_key
   AND llaves.planta_key = disponibilidad.planta_key

LEFT JOIN observatorio_dev.gold.dim_fecha calendario
    ON llaves.fecha_key = calendario.fecha_key

LEFT JOIN observatorio_dev.gold.dim_planta planta
    ON llaves.planta_key = planta.planta_key

## 8. Generación diaria por tipo

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_generacion_diaria_tipo AS

SELECT
    fecha_key,
    fecha,

    MAX(anio) AS anio,
    MAX(trimestre) AS trimestre,
    MAX(mes_numero) AS mes_numero,
    MAX(mes_nombre) AS mes_nombre,
    MAX(anio_mes) AS anio_mes,
    MAX(anio_mes_nombre) AS anio_mes_nombre,

    COALESCE(
        NULLIF(TRIM(tipo_generacion), ''),
        'SIN CLASIFICAR'
    ) AS tipo_generacion,

    ROUND(
        SUM(generacion_total_gwh),
        3
    ) AS generacion_total_gwh,

    ROUND(
        SUM(disponibilidad_total_gwh),
        3
    ) AS disponibilidad_total_gwh,

    COUNT(DISTINCT planta_key)
        AS plantas_con_datos,

    ROUND(
        100.0
        * SUM(generacion_total_gwh)
        / CASE
            WHEN SUM(disponibilidad_total_gwh) = 0
            THEN NULL
            ELSE SUM(disponibilidad_total_gwh)
          END,
        2
    ) AS utilizacion_disponibilidad_pct

FROM observatorio_dev.gold_analytics.vw_operacion_diaria_planta

GROUP BY
    fecha_key,
    fecha,
    COALESCE(
        NULLIF(TRIM(tipo_generacion), ''),
        'SIN CLASIFICAR'
    )

## 9. Energía embalsada diaria por planta

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_energia_embalsada_diaria AS

WITH relaciones AS (
    SELECT
        planta_key,
        COUNT(DISTINCT embalse_key)
            AS cantidad_embalses,
        MAX(
            CASE
                WHEN es_relacion_unica
                THEN embalse_key
            END
        ) AS embalse_key_unico,
        CONCAT_WS(
            ', ',
            SORT_ARRAY(
                COLLECT_SET(codigo_embalse)
            )
        ) AS codigos_embalses_relacionados
    FROM observatorio_dev.gold.bridge_planta_embalse
    GROUP BY planta_key
)

SELECT
    energia.fecha_key,
    energia.fecha_medicion AS fecha,

    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,

    energia.planta_key,
    planta.codigo_planta,
    planta.nombre_planta,
    planta.codigo_sic_agente,

    relaciones.cantidad_embalses,
    relaciones.embalse_key_unico AS embalse_key,
    embalse.codigo_embalse,
    embalse.nombre_embalse,
    embalse.latitud,
    embalse.longitud,

    CASE
        WHEN relaciones.cantidad_embalses = 1
        THEN 'RELACION UNICA'
        WHEN relaciones.cantidad_embalses > 1
        THEN 'RELACION MULTIPLE'
        ELSE 'SIN RELACION'
    END AS estado_relacion,

    relaciones.codigos_embalses_relacionados,

    ROUND(
        energia.energia_embalsada_kwh / 1000000,
        3
    ) AS energia_embalsada_gwh,

    energia.version_seleccionada,
    energia.prioridad_version,

    CASE
        WHEN relaciones.cantidad_embalses = 1
        THEN TRUE
        ELSE FALSE
    END AS es_asignacion_directa

FROM observatorio_dev.gold.fact_energia_embalsada_planta energia

LEFT JOIN observatorio_dev.gold.dim_fecha calendario
    ON energia.fecha_key = calendario.fecha_key

LEFT JOIN observatorio_dev.gold.dim_planta planta
    ON energia.planta_key = planta.planta_key

LEFT JOIN relaciones
    ON energia.planta_key = relaciones.planta_key

LEFT JOIN observatorio_dev.gold.dim_embalse embalse
    ON relaciones.embalse_key_unico = embalse.embalse_key

## 10. Resumen diario de energía embalsada

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_resumen_energia_embalsada_diaria AS

WITH resumen AS (
    SELECT
        fecha_key,
        fecha,

        MAX(anio) AS anio,
        MAX(mes_numero) AS mes_numero,
        MAX(mes_nombre) AS mes_nombre,
        MAX(anio_mes) AS anio_mes,

        SUM(energia_embalsada_gwh)
            AS energia_total_gwh,

        SUM(
            CASE
                WHEN estado_relacion = 'RELACION UNICA'
                THEN energia_embalsada_gwh
                ELSE 0
            END
        ) AS energia_asignacion_directa_gwh,

        SUM(
            CASE
                WHEN estado_relacion = 'RELACION MULTIPLE'
                THEN energia_embalsada_gwh
                ELSE 0
            END
        ) AS energia_relacion_multiple_gwh,

        SUM(
            CASE
                WHEN estado_relacion = 'SIN RELACION'
                THEN energia_embalsada_gwh
                ELSE 0
            END
        ) AS energia_sin_relacion_gwh,

        COUNT(DISTINCT planta_key)
            AS plantas_con_medicion

    FROM observatorio_dev.gold_analytics.vw_energia_embalsada_diaria

    GROUP BY
        fecha_key,
        fecha
),

comparacion AS (
    SELECT
        resumen.*,
        LAG(energia_total_gwh) OVER (
            ORDER BY fecha
        ) AS energia_dia_anterior_gwh
    FROM resumen
)

SELECT
    comparacion.*,

    energia_total_gwh
        - energia_dia_anterior_gwh
        AS variacion_diaria_gwh,

    CASE
        WHEN energia_dia_anterior_gwh > 0
        THEN
            100.0
            * (
                energia_total_gwh
                - energia_dia_anterior_gwh
              )
            / energia_dia_anterior_gwh
    END AS variacion_diaria_pct,

    CASE
        WHEN energia_total_gwh > 0
        THEN
            100.0
            * energia_asignacion_directa_gwh
            / energia_total_gwh
    END AS cobertura_asignacion_directa_pct,

    MAX(fecha) OVER ()
        AS fecha_maxima_disponible

FROM comparacion

## 11. Dimensiones simplificadas para Power BI

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_dim_agente_powerbi AS

SELECT
    agente_key,
    codigo_agente,
    nombre_agente,
    nombre_agente_normalizado,
    actividad_agente
FROM observatorio_dev.gold.dim_agente
WHERE es_actual = TRUE

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold_analytics.vw_dim_planta_powerbi AS

SELECT
    planta_key,
    codigo_planta,
    nombre_planta,
    codigo_sic_agente,
    tipo_generacion,
    cap_efectiva_neta,
    es_registro_inferido,
    esta_en_maestro_actual
FROM observatorio_dev.gold.dim_planta

## 12. Validación final de vistas

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    LongType,
)

ANALYTICS_VIEWS = [
    f"{ANALYTICS_SCHEMA}.vw_sistema_horario",
    f"{ANALYTICS_SCHEMA}.vw_resumen_diario_sistema",
    f"{ANALYTICS_SCHEMA}.vw_actualizacion_fuentes",
    f"{ANALYTICS_SCHEMA}.vw_demanda_diaria_mercado",
    f"{ANALYTICS_SCHEMA}.vw_demanda_diaria_agente",
    f"{ANALYTICS_SCHEMA}.vw_operacion_diaria_planta",
    f"{ANALYTICS_SCHEMA}.vw_generacion_diaria_tipo",
    f"{ANALYTICS_SCHEMA}.vw_energia_embalsada_diaria",
    f"{ANALYTICS_SCHEMA}.vw_resumen_energia_embalsada_diaria",
    f"{ANALYTICS_SCHEMA}.vw_dim_agente_powerbi",
    f"{ANALYTICS_SCHEMA}.vw_dim_planta_powerbi",
]

validation_results = []

for view_name in ANALYTICS_VIEWS:
    exists = spark.catalog.tableExists(
        view_name
    )

    if exists:
        try:
            row_count = int(
                spark.table(
                    view_name
                ).count()
            )

            error_message = ""

        except Exception as exc:
            row_count = None
            error_message = str(exc)[:500]

    else:
        row_count = None
        error_message = "La vista no existe."

    approved = bool(
        exists
        and row_count is not None
        and row_count > 0
    )

    validation_results.append(
        (
            str(view_name),
            bool(exists),
            row_count,
            str(error_message),
            approved,
        )
    )


validation_schema = StructType([
    StructField(
        "vista",
        StringType(),
        False,
    ),

    StructField(
        "existe",
        BooleanType(),
        False,
    ),

    StructField(
        "filas",
        LongType(),
        True,
    ),

    StructField(
        "error",
        StringType(),
        True,
    ),

    StructField(
        "aprobada",
        BooleanType(),
        False,
    ),
])


validation_df = spark.createDataFrame(
    validation_results,
    schema=validation_schema,
)


display(
    validation_df
    .orderBy("vista")
)


failed_views = (
    validation_df
    .filter(
        ~F.col("aprobada")
    )
    .count()
)


print(
    "Vistas evaluadas:",
    len(ANALYTICS_VIEWS),
)

print(
    "Vistas fallidas:",
    failed_views,
)


if failed_views > 0:
    display(
        validation_df
        .filter(
            ~F.col("aprobada")
        )
        .orderBy("vista")
    )

    raise ValueError(
        "La creación o validación de "
        "las vistas analíticas falló."
    )


print(
    "GOLD ANALYTICS APROBADO.")

## 13. Inventario final

In [0]:
%sql
USE CATALOG observatorio_dev;
SHOW VIEWS IN gold_analytics